In [27]:
# --- Import libraries ---
import os
import json
import pandas as pd
import requests
from typing import List, Dict, Any, Optional
from pathlib import Path
from dotenv import load_dotenv
import time

# Import model-specific libraries
import google.generativeai as genai
from openai import OpenAI
from anthropic import Anthropic

print("All libraries imported successfully!")

All libraries imported successfully!


In [28]:
class LLM_V1:
    """Enhanced LLM class for processing filtered JSON documents from preprocessing pipeline."""
    
    # Class variable to track all initialized instances
    _instances = {}
    
    def __init__(self, model_type: str, model_name: str, prompt: str, instance_id: str = None, **kwargs):
        """
        Initialize LLM with specified model and configuration.
        
        Args:
            model_type: Type of model ('gemini', 'claude', 'deepseek', 'openai')
            model_name: Specific model name
            prompt: Prompt template with {{DOCUMENTATION}} placeholder
            instance_id: Unique identifier for this instance (for filename generation)
            **kwargs: Additional model-specific configuration
        """
        self.model_type = model_type.lower()
        self.model_name = model_name
        self.prompt = prompt
        self.config = kwargs
        
        # Set instance identifier for filename generation
        if instance_id:
            self.instance_id = instance_id
        else:
            self.instance_id = self.model_type
        
        # Register this instance
        LLM_V1._instances[self.instance_id] = self
        
        # Initialize model-specific configurations
        self._setup_model()
        
    def _setup_model(self):
        """Setup model-specific configurations and API clients."""
        if self.model_type == 'gemini':
            self._setup_gemini()
        elif self.model_type == 'claude':
            self._setup_claude()
        elif self.model_type == 'deepseek':
            self._setup_deepseek()
        elif self.model_type == 'openai':
            self._setup_openai()
        else:
            raise ValueError(f"Unsupported model type: {self.model_type}")
    
    def _setup_gemini(self):
        """Setup Gemini API configuration."""
        api_key = os.getenv("GOOGLE_API_KEY")
        if not api_key:
            raise ValueError("GOOGLE_API_KEY not found in environment variables")
        
        genai.configure(api_key=api_key)
        
        # Default generation config
        self.generation_config = {
            "max_output_tokens": self.config.get("max_tokens", 4000),
            "temperature": self.config.get("temperature", 0.2),
            "top_p": self.config.get("top_p", 0.95),
            "top_k": self.config.get("top_k", 40)
        }
        
    def _setup_claude(self):
        """Setup Claude API configuration using official Anthropic SDK."""
        api_key = self.config.get("api_key") or os.getenv("ANTHROPIC_API_KEY")
        if not api_key:
            raise ValueError("ANTHROPIC_API_KEY not found")
        
        self.claude_client = Anthropic(api_key=api_key)
        
        # Default configuration
        self.claude_config = {
            "max_tokens": self.config.get("max_tokens", 4000),
            "temperature": self.config.get("temperature", 0.2),
        }
        
    def _setup_deepseek(self):
        """Setup DeepSeek API configuration."""
        self.api_key = self.config.get("api_key") or os.getenv("DEEPSEEK_API_KEY")
        if not self.api_key:
            raise ValueError("DEEPSEEK_API_KEY not found")
        
        self.api_url = "https://api.deepseek.com/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
    
    def _setup_openai(self):
        """Setup OpenAI API configuration."""
        api_key = self.config.get("api_key") or os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY not found")
        
        self.client = OpenAI(api_key=api_key)
        
        # Default configuration
        self.openai_config = {
            "max_tokens": self.config.get("max_tokens", 4000),
            "temperature": self.config.get("temperature", 0.2),
            "top_p": self.config.get("top_p", 0.95),
            "frequency_penalty": self.config.get("frequency_penalty", 0),
            "presence_penalty": self.config.get("presence_penalty", 0)
        }
    
    def query(self, content: str, max_tokens: Optional[int] = None) -> str:
        """
        Query the LLM with given content.
        
        Args:
            content: Text content to process
            max_tokens: Maximum tokens for response (overrides default)
            
        Returns:
            Model response as string
        """
        # Format prompt with content
        formatted_prompt = self.prompt.replace("{{DOCUMENTATION}}", content)
        
        if self.model_type == 'gemini':
            return self._query_gemini(formatted_prompt, max_tokens)
        elif self.model_type == 'claude':
            return self._query_claude(formatted_prompt, max_tokens)
        elif self.model_type == 'deepseek':
            return self._query_deepseek(formatted_prompt, max_tokens)
        elif self.model_type == 'openai':
            return self._query_openai(formatted_prompt, max_tokens)
    
    def _query_gemini(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """Query Gemini model."""
        try:
            config = self.generation_config.copy()
            if max_tokens:
                config["max_output_tokens"] = max_tokens
            
            model = genai.GenerativeModel(
                model_name=self.model_name,
                generation_config=config
            )
            
            response = model.generate_content(prompt)
            return response.text
            
        except Exception as e:
            raise Exception(f"Gemini query failed: {str(e)}")
    
    def _query_claude(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """Query Claude model using official Anthropic SDK."""
        try:
            config = self.claude_config.copy()
            if max_tokens:
                config["max_tokens"] = max_tokens
            
            response = self.claude_client.messages.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                **config
            )
            
            return response.content[0].text
            
        except Exception as e:
            raise Exception(f"Claude query failed: {str(e)}")
    
    def _query_deepseek(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """Query DeepSeek model."""
        try:
            if not prompt or not isinstance(prompt, str) or not prompt.strip():
                raise ValueError("Prompt for DeepSeek cannot be empty.")
            # Separate the prompt template and the input document
            # Assume the prompt template contains '{{DOCUMENTATION}}'
            if "{{DOCUMENTATION}}" in self.prompt:
                # Extract the input document part
                input_doc = prompt.split(self.prompt.replace("{{DOCUMENTATION}}", "")).pop() if self.prompt.replace("{{DOCUMENTATION}}", "") in prompt else prompt
                prompt_prefix = self.prompt.split("{{DOCUMENTATION}}")[0]
                prompt_suffix = self.prompt.split("{{DOCUMENTATION}}")[1] if self.prompt.count("{{DOCUMENTATION}}") == 1 else ""
                # Estimate token budget for input document
                max_context_tokens = 65000
                max_completion_tokens = max_tokens or self.config.get("max_tokens", 4000)
                # Estimate input tokens (very rough: len(prompt)//4)
                est_prompt_tokens = (len(prompt_prefix) + len(prompt_suffix)) // 4
                est_input_tokens = len(input_doc) // 4
                allowed_input_tokens = max_context_tokens - max_completion_tokens - est_prompt_tokens
                if est_input_tokens > allowed_input_tokens:
                    allowed_chars = allowed_input_tokens * 4
                    print(f"[DeepSeek] Truncating input document from {len(input_doc)} to {allowed_chars} characters to fit context window.")
                    input_doc = input_doc[:allowed_chars]
                # Reconstruct prompt with truncated input_doc
                prompt = f"{prompt_prefix}{input_doc}{prompt_suffix}"
            data = {
                "model": self.model_name,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": max_tokens or self.config.get("max_tokens", 4000)
            }
            print("[DeepSeek] Payload:", json.dumps(data, indent=2)[:500])  # Debug: show payload
            response = requests.post(self.api_url, headers=self.headers, json=data)
            try:
                response.raise_for_status()
            except requests.HTTPError as e:
                print("[DeepSeek] Error response:", response.text[:500])  # Debug: show error body
                raise
            return response.json()["choices"][0]["message"]["content"]
        except Exception as e:
            raise Exception(f"DeepSeek query failed: {str(e)}")
    
    def _query_openai(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """Query OpenAI model."""
        try:
            config = self.openai_config.copy()
            if max_tokens:
                config["max_tokens"] = max_tokens
            
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                **config
            )
            
            return response.choices[0].message.content
            
        except Exception as e:
            raise Exception(f"OpenAI query failed: {str(e)}")
    
    def process_filtered_document(self, json_path: str, output_dir: str, filename: str = None) -> None:
        """
        Process a filtered JSON document containing spatial content.
        
        Args:
            json_path: Path to JSON file with filtered document data
            output_dir: Directory to save results
            filename: Base filename for output (without extension)
        """
        try:
            # Load JSON data
            with open(json_path, 'r', encoding='utf-8') as f:
                doc_data = json.load(f)
            
            # Extract document ID and content
            doc_id = doc_data.get('doc_id', Path(json_path).stem)
            filtered_content = doc_data.get('filtered_document', '')
            
            if not filtered_content:
                print(f"❌ No filtered_document content found in {json_path}")
                return
            
            # Construct output filename
            if filename is None:
                filename = f"{self.instance_id}_{doc_id}_spatial_analysis"
            
            if not filename.endswith('.csv'):
                filename = f"{filename}.csv"
                
            output_path = os.path.join(output_dir, filename)
            
            # Check if output file already exists
            if os.path.exists(output_path):
                print(f"⚠ Output file {filename} already exists. Skipping processing.")
                return
            
            # Ensure output directory exists
            os.makedirs(output_dir, exist_ok=True)
            
            print(f"✓ Processing document {doc_id}")
            print(f"  Original length: {doc_data.get('original_length', 'unknown')} characters")
            print(f"  Filtered length: {doc_data.get('filtered_length', len(filtered_content))} characters")
            print(f"  Spatial sections: {doc_data.get('spatial_sections', 'unknown')}")
            print(f"  Sections with spatial indicators: {doc_data.get('sections_with_spatial_indicators', 'unknown')}")
            
            try:
                # Query the model with the filtered content
                model_output = self.query(filtered_content)
                
                # Try to parse JSON output for success tracking
                parsed_successfully = self._check_json_parsing(model_output)
                
                # Create result record
                result = {
                    "document_id": doc_id,
                    "model_type": self.model_type,
                    "model_name": self.model_name,
                    "original_length": doc_data.get('original_length', 0),
                    "filtered_length": doc_data.get('filtered_length', len(filtered_content)),
                    "spatial_sections": doc_data.get('spatial_sections', 0),
                    "sections_with_spatial_indicators": doc_data.get('sections_with_spatial_indicators', 0),
                    "parsed_successfully": parsed_successfully,
                    "raw_output": model_output.strip()
                }
                
                # Save result to CSV
                self._save_results_to_csv([result], output_path)
                print(f"✅ Processing complete. Results saved to {output_path}")
                
            except Exception as e:
                # Save error result
                error_result = {
                    "document_id": doc_id,
                    "model_type": self.model_type,
                    "model_name": self.model_name,
                    "original_length": doc_data.get('original_length', 0),
                    "filtered_length": doc_data.get('filtered_length', len(filtered_content)),
                    "spatial_sections": doc_data.get('spatial_sections', 0),
                    "sections_with_spatial_indicators": doc_data.get('sections_with_spatial_indicators', 0),
                    "parsed_successfully": False,
                    "raw_output": f"Error: {str(e)}"
                }
                
                self._save_results_to_csv([error_result], output_path)
                print(f"❌ Error during processing: {str(e)}")
                
        except Exception as e:
            print(f"❌ Error processing document: {e}")
    
    def process_filtered_directory(self, filtered_docs_dir: str, output_dir: str = None, 
                                 preview_only: bool = False) -> None:
        """
        Process all filtered JSON documents in a directory.
        
        Args:
            filtered_docs_dir: Directory containing filtered JSON files
            output_dir: Output directory (defaults to filtered_docs_dir/llm_results)
            preview_only: If True, only show what would be processed
        """
        filtered_docs_dir = Path(filtered_docs_dir)
        
        if output_dir is None:
            output_dir = filtered_docs_dir / "llm_results"
        else:
            output_dir = Path(output_dir)
        
        # Create output directory
        output_dir.mkdir(exist_ok=True)
        
        # Find JSON files with complete data (containing filtered_document)
        json_files = []
        for json_file in filtered_docs_dir.glob("*_complete.json"):
            json_files.append(json_file)
        
        # Also look for metadata files that might contain filtered_document
        for json_file in filtered_docs_dir.glob("*_metadata.json"):
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                if 'filtered_document' in data and data['filtered_document']:
                    json_files.append(json_file)
            except:
                continue
        
        if not json_files:
            print(f"❌ No JSON files with filtered_document content found in {filtered_docs_dir}")
            return
        
        # Check which files would be processed vs skipped
        files_to_process = []
        files_to_skip = []
        
        for json_file in json_files:
            # Extract document ID from filename
            doc_id = json_file.stem.replace('_metadata', '').replace('_complete', '').replace('test_', '')
            
            output_filename = f"{self.instance_id}_{doc_id}_spatial_analysis.csv"
            output_path = output_dir / output_filename
            
            if output_path.exists():
                files_to_skip.append((json_file.name, output_filename))
            else:
                files_to_process.append((json_file, doc_id, output_filename))
        
        # Display summary
        print(f"\n{'='*60}")
        print(f"📋 SPATIAL ANALYSIS PLAN FOR {self.instance_id.upper()}")
        print(f"{'='*60}")
        print(f"📁 Input Directory: {filtered_docs_dir}")
        print(f"💾 Output Directory: {output_dir}")
        
        print(f"\n✅ DOCUMENTS TO PROCESS ({len(files_to_process)}):")
        if files_to_process:
            for json_file, doc_id, output_file in files_to_process:
                print(f"   Doc {doc_id}: {json_file.name} -> {output_file}")
        else:
            print("   None")
        
        print(f"\n⏭️  DOCUMENTS TO SKIP ({len(files_to_skip)}) - Already processed:")
        if files_to_skip:
            for json_file, output_file in files_to_skip:
                print(f"   {json_file} -> {output_file} (exists)")
        else:
            print("   None")
        
        print(f"\n📊 SUMMARY:")
        print(f"   JSON files found: {len(json_files)}")
        print(f"   Files to process: {len(files_to_process)}")
        print(f"   Files to skip: {len(files_to_skip)}")
        
        if preview_only:
            print(f"\n👀 PREVIEW MODE - No files will be processed")
            return
        
        if not files_to_process:
            print(f"\n✨ All documents already processed for {self.instance_id}!")
            return
        
        # Ask for confirmation if there are files to process
        estimated_cost = len(files_to_process) * 0.75  # Rough estimate per document
        print(f"\n💰 Estimated cost: ~${estimated_cost:.2f} (rough estimate)")
        
        proceed = input(f"\n❓ Proceed with processing {len(files_to_process)} documents? (y/N): ").lower().strip()
        
        if proceed != 'y':
            print("❌ Processing cancelled by user")
            return
        
        print(f"\n🚀 Starting spatial analysis processing...")
        
        # Process documents one by one
        for i, (json_file, doc_id, output_filename) in enumerate(files_to_process, 1):
            print(f"\n📄 Processing document {i}/{len(files_to_process)}: {doc_id}...")
            try:
                self.process_filtered_document(
                    str(json_file), 
                    str(output_dir), 
                    filename=output_filename
                )
                print(f"✅ Completed {i}/{len(files_to_process)}: {output_filename} saved")
            except Exception as e:
                print(f"❌ Failed document {doc_id}: {str(e)}")
        
        print(f"\n🎉 Spatial analysis processing complete! {len(files_to_process)} documents processed.")
    
    def test_connection(self) -> bool:
        """Test model connection with a simple query."""
        try:
            test_input = "Test connection. Please respond with 'Connection successful.'"
            response = self.query(test_input)
            print(f"✅ {self.instance_id} connection successful")
            return True
        except Exception as e:
            print(f"❌ {self.instance_id} connection failed: {str(e)}")
            return False
    
    def quick_test(self, test_input: str = None) -> None:
        """Quick test with sample spatial content."""
        if test_input is None:
            test_input = """
            # Spatial Requirements Test
            
            ## Wind Turbine Spacing
            Wind turbines shall be spaced a minimum of 1.15 miles (1 nautical mile) apart in both east-west and north-south directions.
            
            ## Setback Requirements
            All offshore wind structures must maintain a setback of at least 500 meters from designated shipping lanes.
            """
        
        print(f"\n=== Testing {self.instance_id} ===")
        try:
            response = self.query(test_input)
            print(f"✅ Response received ({len(response)} characters)")
            print(f"Preview: {response[:200]}...")
        except Exception as e:
            print(f"❌ Test failed: {str(e)}")
    
    # Static methods for multi-model operations
    @staticmethod
    def get_working_models():
        """Get all working model instances."""
        working_models = {}
        for name, instance in LLM_V1._instances.items():
            if instance.test_connection():
                working_models[name.title()] = instance
        return working_models
    
    @staticmethod
    def test_all_connections():
        """Test all model connections."""
        print("Testing model connections for spatial analysis...\n")
        
        working_models = {}
        for name, instance in LLM_V1._instances.items():
            if instance.test_connection():
                working_models[name.title()] = instance
            print()
        
        print(f"Working models: {list(working_models.keys())}")
        return working_models
    
    @staticmethod
    def process_with_all_models(json_path: str, output_dir: str = "spatial_results"):
        """Process a filtered JSON document with all working models."""
        working_models = LLM_V1.get_working_models()
        
        json_path = Path(json_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(exist_ok=True)
        
        # Extract document ID
        doc_id = json_path.stem.replace('_metadata', '').replace('_complete', '').replace('test_', '')
        
        print(f"Processing document {doc_id} with all working models...\n")
        
        for name, model in working_models.items():
            filename = f"{name.lower()}_{doc_id}_spatial_analysis"
            output_path = output_dir / f"{filename}.csv"
            
            if output_path.exists():
                print(f"✅ Output file for {name} already exists: {output_path}")
                continue
                
            print(f"📄 Processing with {name}...")
            try:
                model.process_filtered_document(str(json_path), str(output_dir), filename)
                print(f"✅ {name} processing complete - saved to {output_path}")
            except Exception as e:
                print(f"❌ {name} processing failed: {str(e)}")
            print()
    
    # Helper methods
    def _check_json_parsing(self, output: str) -> bool:
        """Check if output contains valid JSON structure."""
        try:
            if "```json" in output:
                json_str = output.split("```json")[1].split("```")[0].strip()
                json.loads(json_str)
                return True
            else:
                json.loads(output)
                return True
        except (json.JSONDecodeError, IndexError):
            return False
    
    def _save_results_to_csv(self, results: List[Dict], output_path: str) -> None:
        """Save processing results to CSV with proper Excel compatibility."""
        df = pd.DataFrame(results)
        df.to_csv(
            output_path, 
            index=False, 
            quoting=2,
            encoding='utf-8-sig',
            sep=',',
            lineterminator='\n'
        )

print("LLM_V1 class defined successfully!")

LLM_V1 class defined successfully!


In [29]:
# Load environment variables
# load_dotenv("D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\API keys\API_keys.env")

# Mac
load_dotenv("/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/API keys/API_keys.env")

True

In [30]:
# --- Import prompts ---
import sys
sys.path.append("/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Prompts")

In [31]:
# try:
#     from prompt_gemini_V1 import prompt as gemini_prompt
#     print("✓ Gemini prompt imported")
# except ImportError:
#     print("⚠ Gemini prompt not found, using default")
#     gemini_prompt = "Analyze the following spatial requirements document: {{DOCUMENTATION}}"

try:
    from prompt_claude_lite_V1 import prompt as gemini_prompt
    print("✓ Gemini prompt imported")
except ImportError:
    print("⚠ Gemini prompt not found, using default")
    gemini_prompt = "Analyze the following spatial requirements document: {{DOCUMENTATION}}"

try:
    from prompt_claude_lite_V1 import prompt as claude_prompt
    print("✓ Claude prompt imported")
except ImportError:
    print("⚠ Claude prompt not found, using default")
    claude_prompt = "Analyze the following spatial requirements document: {{DOCUMENTATION}}"

# try:
#     from prompt_deepseek_V1 import prompt as deepseek_prompt
#     print("✓ DeepSeek prompt imported")
# except ImportError:
#     print("⚠ DeepSeek prompt not found, using default")
#     deepseek_prompt = "Analyze the following spatial requirements document: {{DOCUMENTATION}}"

try:
    from prompt_claude_lite_V1 import prompt as deepseek_prompt
    print("✓ DeepSeek prompt imported")
except ImportError:
    print("⚠ DeepSeek prompt not found, using default")
    deepseek_prompt = "Analyze the following spatial requirements document: {{DOCUMENTATION}}"

# try:
#     from prompt_gemini_V1 import prompt as openai_prompt
#     print("✓ OpenAI prompt imported")
# except ImportError:
#     print("⚠ OpenAI prompt not found, using default")
#     openai_prompt = "Analyze the following spatial requirements document: {{DOCUMENTATION}}"

try:
    from prompt_claude_lite_V1 import prompt as openai_prompt
    print("✓ OpenAI prompt imported")
except ImportError:
    print("⚠ OpenAI prompt not found, using default")
    openai_prompt = "Analyze the following spatial requirements document: {{DOCUMENTATION}}"

✓ Gemini prompt imported
✓ Claude prompt imported
✓ DeepSeek prompt imported
✓ OpenAI prompt imported


In [32]:
# --- Initialize LLM_V1 instances for spatial analysis ---

# Gemini models
gemini_v1 = LLM_V1(
    model_type="gemini",
    model_name="gemini-1.5-pro",
    prompt=gemini_prompt,
    instance_id="gemini_v1",
    max_tokens=4000,
    temperature=0.2
)

gemini25_v1 = LLM_V1(
    model_type="gemini",
    model_name="gemini-2.5-pro",
    prompt=gemini_prompt,
    instance_id="gemini25_v1",
    max_tokens=4000,
    temperature=0.2
)

gemini25_flash_v1 = LLM_V1(
    model_type="gemini",
    model_name="gemini-2.5-flash",
    prompt=gemini_prompt,
    instance_id="gemini25_flash_v1",
    max_tokens=4000,
    temperature=0.2
)

# Claude model
claude_v1 = LLM_V1(
    model_type="claude",
    model_name="claude-3-haiku-20240307",
    prompt=claude_prompt,
    instance_id="claude_v1",
    max_tokens=4000,
    temperature=0.2
)

# DeepSeek model
deepseek_v1 = LLM_V1(
    model_type="deepseek",
    model_name="deepseek-chat",
    prompt=deepseek_prompt,
    instance_id="deepseek_v1",
    max_tokens=4000,
    temperature=0.2
)

# OpenAI models
gpt4o_v1 = LLM_V1(
    model_type="openai",
    model_name="gpt-4o",
    prompt=openai_prompt,
    instance_id="gpt4o_v1",
    max_tokens=4000,
    temperature=0.2
)

gpt4o_mini_v1 = LLM_V1(
    model_type="openai",
    model_name="gpt-4o-mini",
    prompt=openai_prompt,
    instance_id="gpt4o_mini_v1",
    max_tokens=4000,
    temperature=0.2
)

print("All LLM_V1 instances created successfully!")
print("Available models for spatial analysis: Gemini-1.5-Pro, Gemini-2.5-Pro, Gemini-2.5-Flash, Claude-3-Haiku, DeepSeek-Chat, GPT-4o, GPT-4o-mini")

All LLM_V1 instances created successfully!
Available models for spatial analysis: Gemini-1.5-Pro, Gemini-2.5-Pro, Gemini-2.5-Flash, Claude-3-Haiku, DeepSeek-Chat, GPT-4o, GPT-4o-mini


In [33]:
# Set paths for filtered documents
# Mac
filtered_docs_directory = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Filtered_Documents"
spatial_analysis_output = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Processed_Results_V1"

# # PC - Use raw strings to avoid escape sequence issues
# filtered_docs_directory = r"D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Filtered_Documents"
# spatial_analysis_output = r"D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Processed_Results_V1"

# Test Connection and Preview

In [34]:
# Test all model connections
working_models = LLM_V1.test_all_connections()

Testing model connections for spatial analysis...

✅ gemini_v1 connection successful

✅ gemini_v1 connection successful

✅ gemini25_v1 connection successful

✅ gemini25_v1 connection successful

✅ gemini25_flash_v1 connection successful

✅ gemini25_flash_v1 connection successful

✅ claude_v1 connection successful

[DeepSeek] Payload: {
  "model": "deepseek-chat",
  "messages": [
    {
      "role": "user",
      "content": "\nYou are an expert AI system specializing in extracting structured data from energy project documentation. Your primary purpose is to build a spatial knowledge base by analyzing the provided document.\n\n<documentation>\n\nYou are an expert AI system specializing in extracting structured data from energy project documentation. Your primary purpose is to build a spatial knowledge base by analyzing the pro
✅ claude_v1 connection successful

[DeepSeek] Payload: {
  "model": "deepseek-chat",
  "messages": [
    {
      "role": "user",
      "content": "\nYou are an exp

In [35]:
# Preview what would be processed (without actually processing)
if working_models:
    # Pick the first working model for preview
    first_model = list(working_models.values())[0]
    first_model.process_filtered_directory(filtered_docs_directory, spatial_analysis_output, preview_only=True)

❌ No JSON files with filtered_document content found in /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Filtered_Documents


# Process Single Document (Test)

### claude

In [36]:
# # Test with a single document (e.g., 49.json)

# # Mac
# test_json = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_Documents_json/12.json"
# output_dir = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_documents_raw_outputs"

# # Load the JSON and concatenate all section contents for LLM input
# with open(test_json, 'r', encoding='utf-8') as f:
#     doc_data = json.load(f)

# # Compose the document string for the prompt (faithful to the prompt template)
# doc_string = ""
# for section in doc_data.get('sections', []):
#     header = section.get('header_line') or section.get('title', '')
#     content = section.get('content', '')
#     if header:
#         doc_string += f"{header}\n"
#     doc_string += f"{content}\n\n"

# if not doc_string.strip():
#     print(f"❌ No spatial content found in {test_json}")
# else:
#     # Process with one model first (e.g., Claude)
#     if 'Claude_V1' in working_models:
#         response = claude_v1.query(doc_string)
#         # Save output to results directory
#         doc_id = doc_data.get('doc_id', Path(test_json).stem)
#         output_filename = f"claude_v1_{doc_id}.csv"
#         output_path = os.path.join(output_dir, output_filename)
#         result = {
#             "document_id": doc_id,
#             "model_type": claude_v1.model_type,
#             "model_name": claude_v1.model_name,
#             "original_length": doc_data.get('original_length', 0),
#             "filtered_length": len(doc_string),
#             "spatial_sections": doc_data.get('spatial_sections', 0),
#             "sections_with_spatial_indicators": doc_data.get('sections_with_spatial_indicators', 0),
#             "parsed_successfully": claude_v1._check_json_parsing(response),
#             "raw_output": response.strip()
#         }
#         pd.DataFrame([result]).to_csv(
#             output_path,
#             index=False,
#             quoting=2,
#             encoding='utf-8-sig',
#             sep=',',
#             lineterminator='\n'
#         )
#         print(f"✅ Claude output saved to {output_path}")

# Process entire folder

### claude

In [37]:
# # --- Batch process all JSON files in a folder with a selected LLM model ---

# # Set input/output directories
# input_dir = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_Documents_json"
# output_dir = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_documents_raw_outputs"

# # Select the LLM model to use (change this line to switch models)
# selected_model = claude_v1  # e.g., claude_v1, gemini_v1, gpt4o_v1, etc.

# from pathlib import Path

# input_dir = Path(input_dir)
# output_dir = Path(output_dir)
# output_dir.mkdir(exist_ok=True)

# json_files = sorted(input_dir.glob("*.json"))

# print(f"Processing {len(json_files)} files with model: {selected_model.instance_id}")

# for json_file in json_files:
#     try:
#         with open(json_file, 'r', encoding='utf-8') as f:
#             doc_data = json.load(f)
#         doc_id = doc_data.get('doc_id', json_file.stem)

#         # Compose the document string for the prompt
#         doc_string = ""
#         for section in doc_data.get('sections', []):
#             header = section.get('header_line') or section.get('title', '')
#             content = section.get('content', '')
#             if header:
#                 doc_string += f"{header}\n"
#             doc_string += f"{content}\n\n"

#         if not doc_string.strip():
#             print(f"❌ No spatial content found in {json_file.name}, skipping.")
#             continue

#         output_filename = f"{selected_model.instance_id}_{doc_id}.csv"
#         output_path = output_dir / output_filename

#         if output_path.exists():
#             print(f"⏭️  {output_filename} already exists, skipping.")
#             continue

#         print(f"🚀 Processing {json_file.name} -> {output_filename}")

#         # --- Retry logic for rate limit errors ---
#         max_retries = 5
#         delay = 10  # seconds
#         for attempt in range(max_retries):
#             try:
#                 response = selected_model.query(doc_string)
#                 break
#             except Exception as e:
#                 err_str = str(e)
#                 if "rate limit" in err_str.lower() or "429" in err_str:
#                     print(f"⚠️ Rate limit hit. Waiting {delay} seconds before retrying (attempt {attempt+1}/{max_retries})...")
#                     time.sleep(delay)
#                     delay *= 2  # Exponential backoff
#                 else:
#                     raise
#         else:
#             print(f"❌ Failed after {max_retries} retries due to rate limits.")
#             continue

#         result = {
#             "document_id": doc_id,
#             "model_type": selected_model.model_type,
#             "model_name": selected_model.model_name,
#             "original_length": doc_data.get('original_length', 0),
#             "filtered_length": len(doc_string),
#             "spatial_sections": doc_data.get('spatial_sections', 0),
#             "sections_with_spatial_indicators": doc_data.get('sections_with_spatial_indicators', 0),
#             "parsed_successfully": selected_model._check_json_parsing(response),
#             "raw_output": response.strip()
#         }

#         pd.DataFrame([result]).to_csv(
#             output_path,
#             index=False,
#             quoting=2,
#             encoding='utf-8-sig',
#             sep=',',
#             lineterminator='\n'
#         )
#         print(f"✅ Saved to {output_path}")

#     except Exception as e:
#         print(f"❌ Error processing {json_file.name}: {e}")

### Deepseek

In [38]:
# --- Batch process all JSON files in a folder with a selected LLM model ---

# Set input/output directories
input_dir = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_Documents_json"
output_dir = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_documents_raw_outputs"

# Select the LLM model to use (change this line to switch models)
selected_model = deepseek_v1  # e.g., claude_v1, gemini_v1, gpt4o_v1, etc.

from pathlib import Path

input_dir = Path(input_dir)
output_dir = Path(output_dir)
output_dir.mkdir(exist_ok=True)

json_files = sorted(input_dir.glob("*.json"))

print(f"Processing {len(json_files)} files with model: {selected_model.instance_id}")

for json_file in json_files:
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            doc_data = json.load(f)
        doc_id = doc_data.get('doc_id', json_file.stem)

        # Compose the document string for the prompt
        doc_string = ""
        for section in doc_data.get('sections', []):
            header = section.get('header_line') or section.get('title', '')
            content = section.get('content', '')
            if header:
                doc_string += f"{header}\n"
            doc_string += f"{content}\n\n"

        if not doc_string.strip():
            print(f"❌ No spatial content found in {json_file.name}, skipping.")
            continue

        output_filename = f"{selected_model.instance_id}_{doc_id}.csv"
        output_path = output_dir / output_filename

        if output_path.exists():
            print(f"⏭️  {output_filename} already exists, skipping.")
            continue

        print(f"🚀 Processing {json_file.name} -> {output_filename}")
        response = selected_model.query(doc_string)

        result = {
            "document_id": doc_id,
            "model_type": selected_model.model_type,
            "model_name": selected_model.model_name,
            "original_length": doc_data.get('original_length', 0),
            "filtered_length": len(doc_string),
            "spatial_sections": doc_data.get('spatial_sections', 0),
            "sections_with_spatial_indicators": doc_data.get('sections_with_spatial_indicators', 0),
            "parsed_successfully": selected_model._check_json_parsing(response),
            "raw_output": response.strip()
        }

        pd.DataFrame([result]).to_csv(
            output_path,
            index=False,
            quoting=2,
            encoding='utf-8-sig',
            sep=',',
            lineterminator='\n'
        )
        print(f"✅ Saved to {output_path}")

    except Exception as e:
        print(f"❌ Error processing {json_file.name}: {e}")

Processing 28 files with model: deepseek_v1
⏭️  deepseek_v1_1.csv already exists, skipping.
⏭️  deepseek_v1_10.csv already exists, skipping.
🚀 Processing 11.json -> deepseek_v1_11.csv
[DeepSeek] Truncating input document from 517302 to 237640 characters to fit context window.
[DeepSeek] Payload: {
  "model": "deepseek-chat",
  "messages": [
    {
      "role": "user",
      "content": "\nYou are an expert AI system specializing in extracting structured data from energy project documentation. Your primary purpose is to build a spatial knowledge base by analyzing the provided document.\n\n<documentation>\n\nYou are an expert AI system specializing in extracting structured data from energy project documentation. Your primary purpose is to build a spatial knowledge base by analyzing the pro
✅ Saved to /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_documents_raw_outputs/deepseek_v1_11.csv
⏭️  deepseek_v1_12.csv already exists, skipping.
⏭️  deepseek

### GPT-4O-MINI

In [39]:
# # --- Batch process all JSON files in a folder with a selected LLM model ---

# # Set input/output directories
# input_dir = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_Documents_json"
# output_dir = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_documents_raw_outputs"

# # Select the LLM model to use (change this line to switch models)
# selected_model = gpt4o_mini_v1  # e.g., claude_v1, gemini_v1, gpt4o_v1, etc.

# from pathlib import Path

# input_dir = Path(input_dir)
# output_dir = Path(output_dir)
# output_dir.mkdir(exist_ok=True)

# json_files = sorted(input_dir.glob("*.json"))

# print(f"Processing {len(json_files)} files with model: {selected_model.instance_id}")

# for json_file in json_files:
#     try:
#         with open(json_file, 'r', encoding='utf-8') as f:
#             doc_data = json.load(f)
#         doc_id = doc_data.get('doc_id', json_file.stem)

#         # Compose the document string for the prompt
#         doc_string = ""
#         for section in doc_data.get('sections', []):
#             header = section.get('header_line') or section.get('title', '')
#             content = section.get('content', '')
#             if header:
#                 doc_string += f"{header}\n"
#             doc_string += f"{content}\n\n"

#         if not doc_string.strip():
#             print(f"❌ No spatial content found in {json_file.name}, skipping.")
#             continue

#         output_filename = f"{selected_model.instance_id}_{doc_id}.csv"
#         output_path = output_dir / output_filename

#         if output_path.exists():
#             print(f"⏭️  {output_filename} already exists, skipping.")
#             continue

#         print(f"🚀 Processing {json_file.name} -> {output_filename}")
#         response = selected_model.query(doc_string)

#         result = {
#             "document_id": doc_id,
#             "model_type": selected_model.model_type,
#             "model_name": selected_model.model_name,
#             "original_length": doc_data.get('original_length', 0),
#             "filtered_length": len(doc_string),
#             "spatial_sections": doc_data.get('spatial_sections', 0),
#             "sections_with_spatial_indicators": doc_data.get('sections_with_spatial_indicators', 0),
#             "parsed_successfully": selected_model._check_json_parsing(response),
#             "raw_output": response.strip()
#         }

#         pd.DataFrame([result]).to_csv(
#             output_path,
#             index=False,
#             quoting=2,
#             encoding='utf-8-sig',
#             sep=',',
#             lineterminator='\n'
#         )
#         print(f"✅ Saved to {output_path}")

#     except Exception as e:
#         print(f"❌ Error processing {json_file.name}: {e}")

### Gemini

In [40]:
# # --- Batch process all JSON files in a folder with a selected LLM model ---

# # Set input/output directories
# input_dir = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_Documents_json"
# output_dir = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_documents_raw_outputs"

# # Select the LLM model to use (change this line to switch models)
# selected_model = gemini_v1  # e.g., claude_v1, gemini_v1, gpt4o_v1, etc.

# from pathlib import Path

# input_dir = Path(input_dir)
# output_dir = Path(output_dir)
# output_dir.mkdir(exist_ok=True)

# json_files = sorted(input_dir.glob("*.json"))

# print(f"Processing {len(json_files)} files with model: {selected_model.instance_id}")

# for json_file in json_files:
#     try:
#         with open(json_file, 'r', encoding='utf-8') as f:
#             doc_data = json.load(f)
#         doc_id = doc_data.get('doc_id', json_file.stem)

#         # Compose the document string for the prompt
#         doc_string = ""
#         for section in doc_data.get('sections', []):
#             header = section.get('header_line') or section.get('title', '')
#             content = section.get('content', '')
#             if header:
#                 doc_string += f"{header}\n"
#             doc_string += f"{content}\n\n"

#         if not doc_string.strip():
#             print(f"❌ No spatial content found in {json_file.name}, skipping.")
#             continue

#         output_filename = f"{selected_model.instance_id}_{doc_id}.csv"
#         output_path = output_dir / output_filename

#         if output_path.exists():
#             print(f"⏭️  {output_filename} already exists, skipping.")
#             continue

#         print(f"🚀 Processing {json_file.name} -> {output_filename}")
#         response = selected_model.query(doc_string)

#         result = {
#             "document_id": doc_id,
#             "model_type": selected_model.model_type,
#             "model_name": selected_model.model_name,
#             "original_length": doc_data.get('original_length', 0),
#             "filtered_length": len(doc_string),
#             "spatial_sections": doc_data.get('spatial_sections', 0),
#             "sections_with_spatial_indicators": doc_data.get('sections_with_spatial_indicators', 0),
#             "parsed_successfully": selected_model._check_json_parsing(response),
#             "raw_output": response.strip()
#         }

#         pd.DataFrame([result]).to_csv(
#             output_path,
#             index=False,
#             quoting=2,
#             encoding='utf-8-sig',
#             sep=',',
#             lineterminator='\n'
#         )
#         print(f"✅ Saved to {output_path}")

#     except Exception as e:
#         print(f"❌ Error processing {json_file.name}: {e}")